# 서울 공공자전거 대여소 3D 복원 (MapAnything → Gaussian Splat)

**런타임 → 런타임 유형 변경 → T4 GPU**로 바꾼 뒤 위에서부터 순서대로 실행하세요.

| 단계 | 내용 | 산출물 |
|---|---|---|
| 1 | GPU 확인, MapAnything 설치 | |
| 2 | `bikeshare_frames_24.zip` 업로드 · 압축 구조/크기 확인 | |
| 3 | (선택) 번호판·얼굴 등 식별정보 영역 블러 | `frames_redacted/`, `redaction_sheet.jpg` |
| 4 | `facebook/map-anything-apache` 추론 (긴 변 축소 + fp16) | |
| 5 | 축별 상·하위 0.5% 이상치 제외, 측정, INRIA PLY 저장 | `results/scene.ply`, `metrics.json`, `scene_meta.json` |
| 6 | splat-transform 3.10.3으로 형식 변환·비교 | `scene.sog`, `scene.spz`, `scene.compressed.ply`, `formats.md` |
| 7 | 결과 ZIP 다운로드 | `bikeshare_results.zip` |

이 노트북의 모든 숫자는 실행할 때 실제로 측정됩니다. 결과는 `results/`에 JSON으로 저장됩니다.

## 1. GPU 확인 및 MapAnything 설치

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import torch
print("torch", torch.__version__, "| cuda", torch.version.cuda, "|", torch.cuda.get_device_name(0))

In [ ]:
%cd /content
!test -d map-anything || git clone --depth 1 https://github.com/facebookresearch/map-anything.git
%cd /content/map-anything
MAPANYTHING_COMMIT = !git rev-parse HEAD
MAPANYTHING_COMMIT = MAPANYTHING_COMMIT[0]
print("map-anything commit:", MAPANYTHING_COMMIT)
!pip install -q -e .
%cd /content
# editable 설치는 커널을 재시작해야 import 경로에 반영되므로, 재시작 없이 바로 쓰도록 경로를 직접 추가
import sys
if "/content/map-anything" not in sys.path:
    sys.path.insert(0, "/content/map-anything")

## 2. 입력 ZIP 업로드 및 구조 확인
`/content/bikeshare_frames_24.zip`이 없으면 업로드 창이 열립니다. 왼쪽 파일 패널에 직접 끌어다 놓아도 됩니다.

In [ ]:
import os, zipfile, glob
from PIL import Image
ZIP = "/content/bikeshare_frames_24.zip"
if not os.path.exists(ZIP):
    from google.colab import files
    up = files.upload()
    os.rename(list(up.keys())[0], ZIP)
with zipfile.ZipFile(ZIP) as z:
    names = [n for n in z.namelist() if not n.endswith("/")]
    print(len(names), "files; folders inside:", sorted({os.path.dirname(n) or "(root)" for n in names}))
    z.extractall("/content/frames_raw")
FRAMES = sorted(glob.glob("/content/frames_raw/**/*.jpg", recursive=True))
sizes = {}
for f in FRAMES:
    with Image.open(f) as im:
        sizes[os.path.basename(f)] = im.size
print("image sizes (W,H):", sorted(set(sizes.values())), "| count:", len(FRAMES))

## 3. (선택) 식별정보 블러
`REDACT_BOXES`는 프레임마다 블러할 영역 `[x0, y0, x1, y1]`입니다. 원본 720×1280 픽셀 기준이며, 사람이 직접 눈으로 확인해 정했습니다.
블러는 복원 **전에** 적용합니다. 그래야 splat 색상에 번호판 같은 식별정보가 남지 않습니다.
실행 후 `redaction_sheet.jpg`를 열어 **빠진 곳이 없는지 반드시 직접 확인**하세요.

In [ ]:
import json, shutil
from PIL import ImageFilter, ImageDraw
REDACT = True
REDACT_BOXES = json.loads(r'''{
 "frame_001.jpg": [{"type":"text","box":[170,200,400,305],"note":"school bus lettering + campus/phone block"},{"type":"person","box":[645,180,705,285],"note":"pedestrian, small/distant"},{"type":"person","box":[585,185,630,255],"note":"distant pedestrian"},{"type":"plate","box":[465,245,560,295],"note":"white car front plate, not legible"}],
 "frame_002.jpg": [{"type":"plate","box":[25,245,132,332],"note":"bus rear plate, legible"},{"type":"text","box":[295,200,535,320],"note":"school name + crest"},{"type":"phone","box":[440,240,525,275],"note":"phone, partly legible"}],
 "frame_003.jpg": [{"type":"plate","box":[28,235,132,325],"note":"bus rear plate, legible"},{"type":"text","box":[300,205,560,320],"note":"school name + crest"},{"type":"phone","box":[455,250,525,285],"note":"phone, partly legible"}],
 "frame_004.jpg": [{"type":"plate","box":[15,225,115,315],"note":"bus rear plate, legible"},{"type":"text","box":[290,200,575,335],"note":"school name + crest"},{"type":"phone","box":[455,245,545,280],"note":"phone, partly legible"},{"type":"person","box":[685,175,720,255],"note":"uncertain"}],
 "frame_005.jpg": [{"type":"text","box":[195,190,520,320],"note":"school name, campus label, crest"},{"type":"phone","box":[390,235,475,285],"note":"phone/website, legible"},{"type":"person","box":[645,170,710,255],"note":"distant person"}],
 "frame_006.jpg": [{"type":"text","box":[170,160,540,300],"note":"school name + crest"},{"type":"phone","box":[395,220,490,265],"note":"phone + website, legible"}],
 "frame_007.jpg": [{"type":"text","box":[140,135,550,275],"note":"school name + crest"},{"type":"phone","box":[395,205,495,260],"note":"phone + website, legible"}],
 "frame_008.jpg": [{"type":"text","box":[165,145,620,275],"note":"school name + crest"},{"type":"phone","box":[445,205,560,265],"note":"phone + website, legible"}],
 "frame_009.jpg": [{"type":"text","box":[190,145,695,285],"note":"school name + crest"},{"type":"phone","box":[500,215,625,275],"note":"phone + website, legible"}],
 "frame_010.jpg": [{"type":"text","box":[180,155,720,295],"note":"school name + crest"},{"type":"phone","box":[520,220,660,285],"note":"phone + website, legible"}],
 "frame_011.jpg": [{"type":"text","box":[300,145,720,285],"note":"school name, website fragment"}],
 "frame_012.jpg": [{"type":"text","box":[555,165,720,295],"note":"partial school lettering"}],
 "frame_013.jpg": [{"type":"person","box":[0,110,70,215],"note":"uncertain: possible rider"}],
 "frame_014.jpg": [{"type":"person","box":[580,60,720,340],"note":"scooter rider, small"},{"type":"plate","box":[270,60,340,110],"note":"car plate, not legible"}],
 "frame_015.jpg": [{"type":"plate","box":[120,230,215,295],"note":"truck plate, partly occluded"},{"type":"text","box":[0,130,145,235],"note":"handwritten vendor sign"},{"type":"plate","box":[610,120,675,170],"note":"distant car plate, not legible"}],
 "frame_016.jpg": [{"type":"plate","box":[592,255,710,317],"note":"truck plate, legible"},{"type":"text","box":[520,150,652,252],"note":"handwritten vendor sign"},{"type":"person","box":[0,20,40,110],"note":"uncertain"}],
 "frame_017.jpg": [{"type":"person","box":[48,32,115,142],"note":"distant pedestrian"},{"type":"person","box":[225,40,310,150],"note":"two distant pedestrians"}],
 "frame_018.jpg": [{"type":"person","box":[145,55,240,160],"note":"two distant pedestrians"},{"type":"text","box":[640,185,720,290],"note":"vendor sign, partial"}],
 "frame_019.jpg": [{"type":"plate","box":[555,285,672,347],"note":"truck plate, legible"},{"type":"text","box":[483,185,615,282],"note":"handwritten vendor sign"},{"type":"text","box":[335,0,478,40],"note":"storefront sign, optional"},{"type":"person","box":[0,45,45,155],"note":"distant pedestrian"},{"type":"person","box":[690,80,720,145],"note":"uncertain"}],
 "frame_020.jpg": [{"type":"plate","box":[485,280,585,340],"note":"truck plate, partly occluded"},{"type":"text","box":[395,178,527,278],"note":"handwritten vendor sign"}],
 "frame_021.jpg": [{"type":"plate","box":[175,230,280,292],"note":"truck plate, partly occluded"},{"type":"text","box":[70,130,205,235],"note":"handwritten vendor sign"}],
 "frame_022.jpg": [{"type":"plate","box":[18,210,112,275],"note":"truck plate, legible"},{"type":"text","box":[0,120,40,210],"note":"edge of vendor sign"}]
}
''')
SRC_DIR = "/content/frames_raw"
IMG_DIR = "/content/frames_redacted" if REDACT else SRC_DIR
if REDACT:
    shutil.rmtree(IMG_DIR, ignore_errors=True); os.makedirs(IMG_DIR)
    thumbs = []
    for f in FRAMES:
        name = os.path.basename(f)
        im = Image.open(f).convert("RGB")
        for item in REDACT_BOXES.get(name, []):
            x0, y0, x1, y1 = [int(v) for v in item["box"]]
            x0, y0 = max(0, x0), max(0, y0); x1, y1 = min(im.width, x1), min(im.height, y1)
            region = im.crop((x0, y0, x1, y1))
            small = region.resize((max(1, region.width // 16), max(1, region.height // 16)))
            im.paste(small.resize(region.size, Image.NEAREST).filter(ImageFilter.GaussianBlur(6)), (x0, y0))
        im.save(os.path.join(IMG_DIR, name), quality=95)
        t = im.copy(); t.thumbnail((180, 320)); ImageDraw.Draw(t).text((4, 4), name, fill=(255, 0, 0)); thumbs.append(t)
    sheet = Image.new("RGB", (180 * 6, 320 * 4), "white")
    for i, t in enumerate(thumbs):
        sheet.paste(t, ((i % 6) * 180, (i // 6) * 320))
    sheet.save("/content/redaction_sheet.jpg"); display(sheet)
print("reconstruction input:", IMG_DIR, "| redacted regions:", sum(len(v) for v in REDACT_BOXES.values()) if REDACT else 0)

## 4. 보조 스크립트 작성 (저장소 `scripts/` 와 동일한 코드)

In [ ]:
%%writefile /content/splat_utils.py
"""MapAnything 출력 → 측정값 + INRIA Gaussian Splat PLY 변환 유틸리티.

numpy만 사용하므로 Colab과 로컬 양쪽에서 동작합니다.
좌표계: MapAnything의 world 좌표(OpenCV: +X 오른쪽, +Y 아래, +Z 앞)를 그대로 PLY에 저장합니다.
뷰어(three.js, +Y 위)에서는 메시를 X축 기준 180° 회전(F = diag(1,-1,-1))해서 보여줍니다.
"""
from __future__ import annotations

import json
import math

import numpy as np

SH_C0 = 0.28209479177387814  # 0차 구면조화 상수: f_dc = (rgb - 0.5) / SH_C0
FLIP = np.diag([1.0, -1.0, -1.0])  # OpenCV world → three.js 표시 좌표


def trim_outliers(pts: np.ndarray, pct: float = 0.5):
    """좌표축(x, y, z)별 상·하위 pct% 를 벗어나는 점을 제외하는 마스크를 반환."""
    lo = np.percentile(pts, pct, axis=0)
    hi = np.percentile(pts, 100.0 - pct, axis=0)
    keep = np.all((pts >= lo) & (pts <= hi), axis=1)
    return keep, lo, hi


def voxel_downsample(pts: np.ndarray, voxel: float) -> np.ndarray:
    """voxel 크기 격자마다 첫 번째 점만 남기는 인덱스(정렬됨)를 반환. voxel<=0 이면 전체."""
    if voxel <= 0:
        return np.arange(len(pts))
    key = np.floor((pts - pts.min(axis=0)) / voxel).astype(np.int64)
    _, idx = np.unique(key, axis=0, return_index=True)
    idx.sort()
    return idx


def bbox_stats(pts: np.ndarray) -> dict:
    mn, mx = pts.min(axis=0), pts.max(axis=0)
    ext = mx - mn
    return {
        "min": mn.tolist(),
        "max": mx.tolist(),
        "extent": ext.tolist(),
        "diagonal": float(np.linalg.norm(ext)),
        "center": ((mn + mx) / 2).tolist(),
    }


def camera_metrics(cam2world: np.ndarray, scene_pts: np.ndarray) -> dict:
    """카메라 위치와 퍼짐 비(spread ratio) 계산.

    spread_ratio = (카메라 위치 bbox 대각선) / (장면 bbox 대각선)
    path_length  = 프레임 순서대로 이은 카메라 이동 거리 합
    """
    pos = cam2world[:, :3, 3]
    cam_box = bbox_stats(pos)
    scene_box = bbox_stats(scene_pts)
    steps = np.linalg.norm(np.diff(pos, axis=0), axis=1)
    centroid = pos.mean(axis=0)
    return {
        "camera_positions": pos.tolist(),
        "camera_bbox": cam_box,
        "camera_path_length": float(steps.sum()),
        "camera_mean_dist_to_centroid": float(np.linalg.norm(pos - centroid, axis=1).mean()),
        "scene_bbox": scene_box,
        "spread_ratio": float(cam_box["diagonal"] / scene_box["diagonal"]) if scene_box["diagonal"] > 0 else None,
    }


def viewer_cameras(cam2world: np.ndarray, look_dist: float) -> list[dict]:
    """뷰어용 촬영 시점 목록 (three.js 표시 좌표로 변환)."""
    out = []
    for T in cam2world:
        R, p = T[:3, :3], T[:3, 3]
        fwd = R[:, 2]
        up = -R[:, 1]
        out.append({
            "position": (FLIP @ p).tolist(),
            "target": (FLIP @ (p + fwd * look_dist)).tolist(),
            "up": (FLIP @ up).tolist(),
        })
    return out


def write_inria_ply(path: str, xyz: np.ndarray, rgb01: np.ndarray, scale: np.ndarray,
                    opacity: float = 0.9) -> int:
    """INRIA 3DGS 형식(binary_little_endian) PLY 저장. 반환값: 파일 바이트 수.

    필드: x y z nx ny nz f_dc_0..2 opacity scale_0..2 rot_0..3 (float32 17개 = 68 B/점)
    - f_dc: (rgb - 0.5) / SH_C0
    - opacity: logit(opacity)  (렌더러가 sigmoid 적용)
    - scale: log(반지름)        (렌더러가 exp 적용), 등방성
    - rot: 단위 쿼터니언 (w, x, y, z) = (1, 0, 0, 0)
    """
    n = len(xyz)
    names = ["x", "y", "z", "nx", "ny", "nz", "f_dc_0", "f_dc_1", "f_dc_2", "opacity",
             "scale_0", "scale_1", "scale_2", "rot_0", "rot_1", "rot_2", "rot_3"]
    data = np.zeros((n, len(names)), dtype=np.float32)
    data[:, 0:3] = xyz
    data[:, 6:9] = (np.clip(rgb01, 0, 1) - 0.5) / SH_C0
    data[:, 9] = math.log(opacity / (1 - opacity))
    log_s = np.log(np.clip(scale, 1e-6, None)).astype(np.float32)
    data[:, 10:13] = log_s[:, None]
    data[:, 13] = 1.0
    header = "ply\nformat binary_little_endian 1.0\n" f"element vertex {n}\n"
    header += "".join(f"property float {nm}\n" for nm in names) + "end_header\n"
    with open(path, "wb") as f:
        f.write(header.encode("ascii"))
        f.write(data.astype("<f4").tobytes())
    return len(header) + data.nbytes


def read_ply_vertex_count(path: str) -> int:
    with open(path, "rb") as f:
        for _ in range(200):
            line = f.readline().decode("ascii", "replace").strip()
            if line.startswith("element vertex"):
                return int(line.split()[-1])
            if line == "end_header":
                break
    raise ValueError(f"vertex count not found in {path}")


def dump_json(path: str, obj) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

In [ ]:
%%writefile /content/convert_formats.py
"""scene.ply → .compressed.ply / .sog / .spz 변환과 크기·시간·점 개수 측정.

사용법:  python scripts/convert_formats.py results/scene.ply results/
요구사항: Node.js >= 22 (splat-transform 3.10.3의 engines 조건), npx

측정 방식
- 변환 시간: 각 변환 명령의 wall-clock (time.perf_counter). npx 패키지 다운로드 시간을 빼기 위해
  측정 전에 `--help`를 한 번 실행해 캐시를 채운다.
- 점 개수: 각 출력 파일을 다시 splat-transform으로 .ply로 되돌려(round-trip) 헤더의 vertex 수를 읽는다.
- 점당 바이트: 출력 파일 크기 / 출력 점 개수.
SOG는 WebGPU를 사용한다. GPU를 못 쓰면 `-g cpu`로 다시 시도하고, 그 사실을 기록한다.
"""
from __future__ import annotations

import json
import os
import shutil
import subprocess
import sys
import time

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from splat_utils import read_ply_vertex_count  # noqa: E402

PKG = "@playcanvas/splat-transform@3.10.3"
FORMATS = [("compressed.ply", "scene.compressed.ply"), ("sog", "scene.sog"), ("spz", "scene.spz")]


def st_cmd(*args: str) -> list[str]:
    npx = shutil.which("npx") or shutil.which("npx.cmd")
    if not npx:
        sys.exit("npx를 찾을 수 없습니다. Node.js 22 이상을 설치하세요.")
    return [npx, "-y", "-p", PKG, "splat-transform", *args]


def run(cmd: list[str]) -> tuple[float, subprocess.CompletedProcess]:
    t0 = time.perf_counter()
    proc = subprocess.run(cmd, capture_output=True, text=True)
    return time.perf_counter() - t0, proc


def main(src: str, outdir: str) -> None:
    os.makedirs(outdir, exist_ok=True)
    node_ver = subprocess.run(["node", "--version"], capture_output=True, text=True).stdout.strip()
    print("node", node_ver)
    run(st_cmd("--help"))  # npx 캐시 준비 (측정 제외)

    src_n = read_ply_vertex_count(src)
    src_size = os.path.getsize(src)
    rows = [{"format": "ply (INRIA, 원본)", "file": os.path.basename(src), "bytes": src_size,
             "seconds": None, "points": src_n, "bytes_per_point": src_size / src_n, "note": "변환 입력"}]

    rt_dir = os.path.join(outdir, "_roundtrip")
    os.makedirs(rt_dir, exist_ok=True)
    for fmt, name in FORMATS:
        out = os.path.join(outdir, name)
        secs, proc = run(st_cmd("-w", src, out))
        note = ""
        if proc.returncode != 0 and fmt == "sog":
            print("SOG GPU 변환 실패, CPU로 재시도:\n", proc.stderr[-800:])
            secs, proc = run(st_cmd("-w", "-g", "cpu", src, out))
            note = "WebGPU 사용 불가로 -g cpu 사용"
        if proc.returncode != 0:
            print(f"[{fmt}] 실패\n{proc.stdout[-800:]}\n{proc.stderr[-800:]}")
            rows.append({"format": fmt, "file": name, "bytes": None, "seconds": secs, "points": None,
                         "bytes_per_point": None, "note": "변환 실패: " + proc.stderr.strip()[-200:]})
            continue
        size = os.path.getsize(out)
        rt = os.path.join(rt_dir, name.replace(".", "_") + ".ply")
        _, rtp = run(st_cmd("-w", out, rt))
        pts = read_ply_vertex_count(rt) if rtp.returncode == 0 else None
        rows.append({"format": fmt, "file": name, "bytes": size, "seconds": round(secs, 3), "points": pts,
                     "bytes_per_point": (size / pts) if pts else None, "note": note})
        print(f"[{fmt}] {size:,} B, {secs:.2f} s, {pts} pts")
    shutil.rmtree(rt_dir, ignore_errors=True)

    result = {"tool": PKG, "node": node_ver, "input": os.path.basename(src), "rows": rows}
    with open(os.path.join(outdir, "formats.json"), "w", encoding="utf-8") as f:
        json.dump(result, f, ensure_ascii=False, indent=2)

    def fmt_num(v, spec):
        return "-" if v is None else format(v, spec)

    lines = ["| 형식 | 파일 | 크기 (MB) | 변환 시간 (s) | 점 개수 | 점당 바이트 | 비고 |",
             "|---|---|---:|---:|---:|---:|---|"]
    for r in rows:
        mb = None if r["bytes"] is None else r["bytes"] / 1e6
        lines.append(f"| {r['format']} | `{r['file']}` | {fmt_num(mb, '.2f')} | {fmt_num(r['seconds'], '.2f')} | "
                     f"{fmt_num(r['points'], ',')} | {fmt_num(r['bytes_per_point'], '.2f')} | {r['note']} |")
    md = "\n".join(lines) + "\n"
    with open(os.path.join(outdir, "formats.md"), "w", encoding="utf-8") as f:
        f.write(md)
    print(md)


if __name__ == "__main__":
    if len(sys.argv) != 3:
        sys.exit(__doc__)
    main(sys.argv[1], sys.argv[2])

## 5. MapAnything 추론
- 긴 변 축소: `load_images(resize_mode="longest_side", size=LONG_SIDE)`. 기본값은 518이고, ViT patch 14의 배수입니다.
- fp16: T4는 bf16 가속을 지원하지 않아서 `amp_dtype="fp16"`을 명시합니다.
- 메모리가 부족하면 `LONG_SIDE`를 434나 392로 줄이세요.

In [ ]:
import os, time, sys
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import numpy as np, torch
sys.path.insert(0, "/content")
sys.path.insert(0, "/content/map-anything")  # editable 설치 직후(커널 재시작 전)에도 import 되도록
from mapanything.models import MapAnything
from mapanything.utils.image import load_images

LONG_SIDE = 518
AMP_DTYPE = "fp16"
TIMES = {}
t0 = time.perf_counter()
model = MapAnything.from_pretrained("facebook/map-anything-apache").to("cuda").eval()
TIMES["model_load_s"] = time.perf_counter() - t0

t0 = time.perf_counter()
views = load_images(IMG_DIR, resize_mode="longest_side", size=LONG_SIDE)
TIMES["image_load_s"] = time.perf_counter() - t0
INPUT_SHAPE = [int(x) for x in views[0]["true_shape"][0]]  # [H, W]
print(len(views), "views | model input H,W =", INPUT_SHAPE, "|", TIMES)

In [ ]:
torch.cuda.reset_peak_memory_stats(); torch.cuda.synchronize()
t0 = time.perf_counter()
with torch.no_grad():
    preds = model.infer(
        views,
        memory_efficient_inference=True,
        minibatch_size=1,
        use_amp=True,
        amp_dtype=AMP_DTYPE,
        apply_mask=True,
        mask_edges=True,
        apply_confidence_mask=False,
    )
torch.cuda.synchronize()
TIMES["inference_s"] = time.perf_counter() - t0
PEAK_GB = torch.cuda.max_memory_allocated() / 2**30
print(f"inference {TIMES['inference_s']:.1f}s | peak GPU mem {PEAK_GB:.2f} GiB | keys: {sorted(preds[0].keys())}")

## 6. 점 데이터 정리 · 측정 · INRIA PLY 저장
- 마스크(`mask`)를 통과한 픽셀만 사용합니다.
- **좌표축별 상·하위 0.5% 이상치 제외**를 적용합니다 (`trim_outliers`).
- 점이 `MAX_POINTS`보다 많으면 voxel 격자로 줄입니다. 각 단계의 점 개수는 모두 기록합니다.
- Gaussian 반지름은 픽셀 하나의 크기를 3D로 옮긴 값(`depth / fx × SCALE_K`)입니다. 회전은 단위 쿼터니언, 불투명도는 0.9로 둡니다.

In [ ]:
from splat_utils import trim_outliers, voxel_downsample, camera_metrics, viewer_cameras, write_inria_ply, bbox_stats, dump_json, FLIP
SCALE_K = 1.0
MAX_POINTS = 1_000_000
os.makedirs("/content/results", exist_ok=True)

P, C, S, POSES, per_view = [], [], [], [], []
for i, p in enumerate(preds):
    pts = p["pts3d"][0].float().cpu().numpy()
    img = p["img_no_norm"][0].float().cpu().numpy()
    if img.max() > 1.5:
        img = img / 255.0
    depth = p["depth_z"][0].squeeze(-1).float().cpu().numpy()
    fx = float(p["intrinsics"][0][0, 0])
    m = p["mask"][0].squeeze(-1).cpu().numpy().astype(bool) & np.isfinite(pts).all(-1) & (depth > 0)
    P.append(pts[m]); C.append(img[m]); S.append(depth[m] / fx * SCALE_K)
    POSES.append(p["camera_poses"][0].float().cpu().numpy())
    per_view.append({"frame": os.path.basename(FRAMES[i]), "pixels": int(m.size), "valid": int(m.sum()), "fx": fx})
P, C, S, POSES = np.concatenate(P), np.concatenate(C), np.concatenate(S), np.stack(POSES)
n_masked = len(P)

keep, lo, hi = trim_outliers(P, 0.5)
P, C, S = P[keep], C[keep], S[keep]
n_trimmed = len(P)

diag = bbox_stats(P)["diagonal"]
voxel = 0.0
if len(P) > MAX_POINTS:
    voxel = diag / 4000
    while True:
        idx = voxel_downsample(P, voxel)
        if len(idx) <= MAX_POINTS: break
        voxel *= 1.25
    P, C, S = P[idx], C[idx], np.maximum(S[idx], voxel * 0.5)
n_final = len(P)

t0 = time.perf_counter()
ply_bytes = write_inria_ply("/content/results/scene.ply", P, C, S)
TIMES["ply_write_s"] = time.perf_counter() - t0

cam = camera_metrics(POSES, P)
metrics = {
    "gpu": torch.cuda.get_device_name(0), "torch": torch.__version__,
    "model": "facebook/map-anything-apache", "mapanything_commit": MAPANYTHING_COMMIT,
    "frames": len(views), "original_size_wh": sorted(set(sizes.values())), "model_input_hw": INPUT_SHAPE,
    "long_side": LONG_SIDE, "amp_dtype": AMP_DTYPE, "redacted": REDACT,
    "timings_s": {k: round(v, 3) for k, v in TIMES.items()}, "peak_gpu_mem_gib": round(PEAK_GB, 3),
    "points": {"masked_valid": n_masked, "after_trim_0.5pct": n_trimmed, "final": n_final,
               "trim_removed": n_masked - n_trimmed, "voxel_size": voxel},
    "trim_bounds": {"low": lo.tolist(), "high": hi.tolist()},
    "per_view": per_view, "ply_bytes": ply_bytes,
    "units_note": "MapAnything 출력 단위(메트릭 스케일 추정, 미터 근사). 실측 기준과 비교하지 않았음.",
    **cam,
}
dump_json("/content/results/metrics.json", metrics)

center = np.array(cam["scene_bbox"]["center"])
dump_json("/content/results/scene_meta.json", {
    "target": (FLIP @ center).tolist(),
    "cameras": viewer_cameras(POSES, look_dist=diag * 0.25),
    "frames": [v["frame"] for v in per_view],
    "points": n_final, "scene_diagonal": diag, "spread_ratio": cam["spread_ratio"],
})
print(json.dumps({k: metrics[k] for k in ["points", "timings_s", "peak_gpu_mem_gib", "spread_ratio"]}, indent=1, ensure_ascii=False))
print("scene bbox extent:", np.round(cam["scene_bbox"]["extent"], 3), "| camera bbox extent:", np.round(cam["camera_bbox"]["extent"], 3))

In [ ]:
# 위에서 내려다본 점 분포 + 카메라 궤적 (README용 미리보기)
import matplotlib.pyplot as plt
sel = np.random.default_rng(0).choice(len(P), min(150_000, len(P)), replace=False)
pos = np.array(cam["camera_positions"])
fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(P[sel, 0], P[sel, 2], c=C[sel], s=0.3, linewidths=0)
ax.plot(pos[:, 0], pos[:, 2], "r.-", lw=1); ax.text(pos[0, 0], pos[0, 2], " 1", color="r")
ax.set_xlabel("x"); ax.set_ylabel("z (forward)"); ax.set_aspect("equal"); ax.set_title("top view: points + cameras")
plt.savefig("/content/results/preview_topdown.png", dpi=120, bbox_inches="tight"); plt.show()

## 7. splat-transform 3.10.3 형식 변환 · 비교
splat-transform 3.10.3은 Node.js 22 이상이 필요합니다. 그래서 Colab에 Node 22를 설치합니다.
SOG 압축은 WebGPU를 사용하고, 실패하면 자동으로 `-g cpu`로 다시 시도합니다. 이때 비고 칸에 그 사실이 기록됩니다.

In [ ]:
!node --version || true
!curl -fsSL https://deb.nodesource.com/setup_22.x | bash - > /dev/null 2>&1 && apt-get install -y -qq nodejs > /dev/null
!node --version
!cd /content && python convert_formats.py results/scene.ply results/

## 8. 결과 다운로드
받은 ZIP을 풀어서 저장소의 `results/` 폴더에 넣으세요. `scene.sog`는 저장소 루트에 둡니다.

In [ ]:
import shutil
if os.path.exists("/content/redaction_sheet.jpg"):
    shutil.copy("/content/redaction_sheet.jpg", "/content/results/")
shutil.make_archive("/content/bikeshare_results", "zip", "/content/results")
print({f: os.path.getsize(os.path.join("/content/results", f)) for f in sorted(os.listdir("/content/results"))})
from google.colab import files
files.download("/content/bikeshare_results.zip")